In [1]:
#Import libraries 
import sys
from pathlib import Path
import re
import json
import random
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import fasttext

from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC


sys.path.append(str(Path.cwd().parent / "src"))
from helpers import pick_by_pos, write_fasttext, train_fasttext, train_svm, eval_metrics_svm, normalize_doc_id, undersample_quantitative, ramp, select_for_iteration_svm

#Set random seeds to ensure reproducibility of results
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

#Experiment parameters
chunk_size = 2000
max_iterations = 30
target_qual_auto = 2000
test_per_class = 20

# Dynamic confidence schedule
TAU_START = 0.70
TAU_END = 0.92
ramp_iteration = max_iterations

# Class imbalance controls
max_quant_to_qual_ratio= 1.5
max_quant_add_per_iteration= 200
undersample_quant_to_qual = 1.5

# Heuristic candidate review sample
heuristic_sample_frac = 0.03

base_dir = Path.cwd().parent
data_dir = base_dir / "data"
output_dir = base_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)

In [2]:
base_train_pool = pd.read_csv(output_dir / "base_train_pool.csv")
eval_df = pd.read_csv(output_dir / "test_df_stage0.csv")
cs_df = pd.read_csv(data_dir / "data.csv")

In [3]:
label_qual = "qualitative"
label_quant = "quantitative"

##Store per-iteration results 
svm_experiment_logs = []
svm_model_registry = []


## Initial Training Pool Setup 

train_pool = base_train_pool[["doc_id", "title", "abstract", "label", "label_norm"]].copy()
train_pool = normalize_doc_id(train_pool)

##Initial Model Training

train_fit = undersample_quantitative(
    train_pool[["doc_id", "title", "abstract", "label"]],
    qual_label=label_qual,
    quant_label=label_quant,
    ratio=undersample_quant_to_qual,
    random_state=RANDOM_SEED,
)

svm_model = train_svm(train_fit)

## Evaluation 

m0 = eval_metrics_svm(svm_model, eval_df, y_col="y_true")
svm_experiment_logs.append({
    "iter": 0,
    "tau": np.nan,
    "margin": np.nan,
    "chunk_size": 0,
    "auto_total_iter": 0,
    "manual_total_iter": 0,
    "qual_auto_iter": 0,
    "quant_auto_iter": 0,
    "added_unique": 0,
    "added_qual_unique": 0,
    "added_quant_unique": 0,
    "train_pool_size": int(len(train_pool)),
    "fit_train_size": int(len(train_fit)),
    "added_qual_cum": 0,
    "acc": round(m0["acc"], 4) if pd.notna(m0["acc"]) else np.nan,
    "f1_macro": round(m0["f1_macro"], 4) if pd.notna(m0["f1_macro"]) else np.nan,
    "f1_qual": round(m0["f1_qual"], 4) if pd.notna(m0["f1_qual"]) else np.nan,
    "f1_quant": round(m0["f1_quant"], 4) if pd.notna(m0["f1_quant"]) else np.nan,
    "precision_macro": round(m0["precision_macro"], 4) if pd.notna(m0["precision_macro"]) else np.nan,
    "recall_macro": round(m0["recall_macro"], 4) if pd.notna(m0["recall_macro"]) else np.nan,
})
svm_model_registry.append({
    "iter": 0,
    "fit_train_size": int(len(train_fit)),
    "train_pool_size": int(len(train_pool)),
    "acc": m0["acc"],
    "f1_macro": m0["f1_macro"],
})

# Build Unlabelled Pool
seen_ids = set(train_pool["doc_id"].astype(str))
unlabeled_pool = cs_df.copy()
unlabeled_pool["doc_id"] = unlabeled_pool["doc_id"].astype(str).str.strip()
unlabeled_pool = unlabeled_pool[~unlabeled_pool["doc_id"].isin(seen_ids)].copy().reset_index(drop=True)

added_qual_cum = 0


###############################################
# Iterative Training Loop 
###############################################

for it in range(1, max_iterations + 1):
    if unlabeled_pool.empty:
        break
    
    ##Tresholding ramping 

    tau_it = ramp(it, TAU_START, TAU_END, ramp_iteration)

    ##Select Chunk of Unlabelled Data
    chunk = unlabeled_pool.iloc[:chunk_size].copy()
    unlabeled_pool = unlabeled_pool.iloc[chunk_size:].copy()

    ## Select predictions for auto labelling or manual labelling 
    auto, manual = select_for_iteration_svm(svm_model, chunk)

    ##Split auto labelled by class
    qual_auto = auto[auto["pred_norm"] == label_qual].copy()
    quant_auto = auto[auto["pred_norm"] == label_quant].copy()

    ##Build new training row
    new_rows = pd.concat([
        qual_auto.assign(label=label_qual)[["doc_id", "title", "abstract", "label"]],
        quant_auto.assign(label=label_quant)[["doc_id", "title", "abstract", "label"]],
    ], ignore_index=True)

    new_rows = normalize_doc_id(new_rows)

    ## Remove duplicates
    existing_ids = set(train_pool["doc_id"].astype(str))
    truly_new = new_rows[~new_rows["doc_id"].astype(str).isin(existing_ids)].copy()

    truly_new["label_norm"] = truly_new["label"].astype(str).str.strip().str.lower()
    truly_new_qual = truly_new[truly_new["label_norm"] == label_qual].copy()
    truly_new_quant = truly_new[truly_new["label_norm"] == label_quant].copy()

    ## Control Class Balance
    cur_qual = int((train_pool["label_norm"] == label_qual).sum())
    cur_quant = int((train_pool["label_norm"] == label_quant).sum())

    projected_qual = cur_qual + len(truly_new_qual)
    quant_allowed_by_ratio = int(max_quant_to_qual_ratio * max(projected_qual, 1)) - cur_quant
    quant_allowed = max(0, min(quant_allowed_by_ratio, max_quant_add_per_iteration))

    ##Limit Quantitative Addition

    if len(truly_new_quant) > quant_allowed:
        if "confidence" in auto.columns:
            conf_map = auto.set_index("doc_id")["confidence"]
            truly_new_quant["confidence"] = truly_new_quant["doc_id"].map(conf_map)
            truly_new_quant = truly_new_quant.sort_values("confidence", ascending=False).head(quant_allowed).copy()
        else:
            truly_new_quant = truly_new_quant.head(quant_allowed).copy()

    #New training data

    final_new = pd.concat([truly_new_qual, truly_new_quant], ignore_index=True)
    added = len(final_new)
    added_qual = len(truly_new_qual)
    added_quant = len(truly_new_quant)

    #Update training pool and retrain model 
    if added > 0:
        train_pool = pd.concat([train_pool, final_new], ignore_index=True)
        train_pool = train_pool.drop_duplicates(subset="doc_id", keep="first").copy()
        train_pool["label_norm"] = train_pool["label"].astype(str).str.strip().str.lower()

        train_fit = undersample_quantitative(
            train_pool[["doc_id", "title", "abstract", "label"]],
            qual_label=label_qual,
            quant_label=label_quant,
            ratio=undersample_quant_to_qual,
            random_state=RANDOM_SEED,
        )
        fit_train_size = len(train_fit)

        svm_model = train_svm(train_fit)
    else:
        fit_train_size = np.nan

    ##Evaluate updated model
    m = eval_metrics_svm(svm_model, eval_df, y_col="y_true")
    added_qual_cum += added_qual

    # Log results
    svm_experiment_logs.append({
        "iter": it,
        "chunk_size": int(len(chunk)),
        "auto_total_iter": int(len(auto)),
        "manual_total_iter": int(len(manual)),
        "qual_auto_iter": int(len(qual_auto)),
        "quant_auto_iter": int(len(quant_auto)),
        "added_unique": int(added),
        "added_qual_unique": int(added_qual),
        "added_quant_unique": int(added_quant),
        "train_pool_size": int(len(train_pool)),
        "fit_train_size": int(fit_train_size) if pd.notna(fit_train_size) else np.nan,
        "added_qual_cum": int(added_qual_cum),
        "acc": round(m["acc"], 4) if pd.notna(m["acc"]) else np.nan,
        "f1_macro": round(m["f1_macro"], 4) if pd.notna(m["f1_macro"]) else np.nan,
        "f1_qual": round(m["f1_qual"], 4) if pd.notna(m["f1_qual"]) else np.nan,
        "f1_quant": round(m["f1_quant"], 4) if pd.notna(m["f1_quant"]) else np.nan,
        "precision_macro": round(m["precision_macro"], 4) if pd.notna(m["precision_macro"]) else np.nan,
        "recall_macro": round(m["recall_macro"], 4) if pd.notna(m["recall_macro"]) else np.nan,
    })
    
    svm_model_registry.append({
        "iter": it,
        "fit_train_size": int(fit_train_size) if pd.notna(fit_train_size) else np.nan,
        "train_pool_size": int(len(train_pool)),
        "acc": m["acc"],
        "f1_macro": m["f1_macro"],
        "f1_qual": m["f1_qual"],
        "f1_quant": m["f1_quant"],
    })
    
    # Print progress
    print(
        f"it={it} chunk={len(chunk)} "
        f"auto={len(auto)} added={added} (+q={added_qual}, +n={added_quant}) "
        f"acc={m['acc']:.4f} "
        f"f1_macro={m['f1_macro']:.4f} "
        f"f1_qual={m['f1_qual']:.4f} "
        f"f1_quant={m['f1_quant']:.4f}"
    )
    
    if added_qual_cum >= target_qual_auto:
        break

it=1 chunk=2000 auto=49 added=2 (+q=2, +n=0) acc=0.8000 f1_macro=0.7980 f1_qual=0.7778 f1_quant=0.8182
it=2 chunk=2000 auto=65 added=4 (+q=4, +n=0) acc=0.8000 f1_macro=0.7980 f1_qual=0.7778 f1_quant=0.8182
it=3 chunk=2000 auto=67 added=3 (+q=3, +n=0) acc=0.8000 f1_macro=0.7980 f1_qual=0.7778 f1_quant=0.8182
it=4 chunk=2000 auto=71 added=3 (+q=3, +n=0) acc=0.8250 f1_macro=0.8222 f1_qual=0.8000 f1_quant=0.8444
it=5 chunk=2000 auto=63 added=3 (+q=3, +n=0) acc=0.8250 f1_macro=0.8222 f1_qual=0.8000 f1_quant=0.8444
it=6 chunk=2000 auto=91 added=4 (+q=4, +n=0) acc=0.8250 f1_macro=0.8222 f1_qual=0.8000 f1_quant=0.8444
it=7 chunk=2000 auto=82 added=3 (+q=3, +n=0) acc=0.8250 f1_macro=0.8222 f1_qual=0.8000 f1_quant=0.8444
it=8 chunk=2000 auto=107 added=1 (+q=1, +n=0) acc=0.8250 f1_macro=0.8222 f1_qual=0.8000 f1_quant=0.8444
it=9 chunk=2000 auto=98 added=3 (+q=3, +n=0) acc=0.8250 f1_macro=0.8222 f1_qual=0.8000 f1_quant=0.8444
it=10 chunk=2000 auto=109 added=4 (+q=4, +n=0) acc=0.8250 f1_macro=0.822

In [4]:
svm_experiment_logs

[{'iter': 0,
  'tau': nan,
  'margin': nan,
  'chunk_size': 0,
  'auto_total_iter': 0,
  'manual_total_iter': 0,
  'qual_auto_iter': 0,
  'quant_auto_iter': 0,
  'added_unique': 0,
  'added_qual_unique': 0,
  'added_quant_unique': 0,
  'train_pool_size': 1155,
  'fit_train_size': 375,
  'added_qual_cum': 0,
  'acc': 0.8,
  'f1_macro': 0.798,
  'f1_qual': 0.7778,
  'f1_quant': 0.8182,
  'precision_macro': 0.8125,
  'recall_macro': 0.8},
 {'iter': 1,
  'chunk_size': 2000,
  'auto_total_iter': 49,
  'manual_total_iter': 1951,
  'qual_auto_iter': 2,
  'quant_auto_iter': 47,
  'added_unique': 2,
  'added_qual_unique': 2,
  'added_quant_unique': 0,
  'train_pool_size': 1157,
  'fit_train_size': 380,
  'added_qual_cum': 2,
  'acc': 0.8,
  'f1_macro': 0.798,
  'f1_qual': 0.7778,
  'f1_quant': 0.8182,
  'precision_macro': 0.8125,
  'recall_macro': 0.8},
 {'iter': 2,
  'chunk_size': 2000,
  'auto_total_iter': 65,
  'manual_total_iter': 1935,
  'qual_auto_iter': 4,
  'quant_auto_iter': 61,
  'add